# C4. Trọng số lớp dịch logit tối ưu một lượng log(a1/a0)

Chứng minh: `proofs/C4.md`. Notebook này kiểm chứng số học bằng `softmax.py` (K = 2, λ = 0, có `sample_weight`);
logit nhị phân f(x) = z₁ − z₀. sklearn `LogisticRegression(class_weight=...)` chỉ dùng để đối chiếu.
Dữ liệu tổng hợp, seed 42–46. Kết quả: `results/C4_logit_shift.csv`.

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
RES = os.path.join(ROOT, "results")
os.makedirs(RES, exist_ok=True)
SEEDS = [42, 43, 44, 45, 46]

import core
from softmax import SoftmaxRegression
from sklearn.linear_model import LogisticRegression

RATIOS = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]   # a1 / a0, giữ a0 = 1


def fit_logit(X, y, w):
    """Fit softmax K = 2, λ = 0; trả về (hàm logit f(X) = z1 − z0, model)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = SoftmaxRegression(lam=0.0, n_classes=2, tol=1e-10, max_iter=50_000).fit(X, y, sample_weight=w)
    f = lambda Xn: (lambda Z: Z[:, 1] - Z[:, 0])(m.decision_function(Xn))
    return f, m


def population_rows(x, px, eta, a1):
    """Mỗi giá trị x thành 2 hàng (y=1, y=0) với trọng số P(x)·η·a1 và P(x)·(1−η)·a0 (a0 = 1).
    Cực tiểu loss có trọng số trên các hàng này = cực tiểu rủi ro tổng thể có trọng số."""
    X = np.concatenate([x, x])
    y = np.concatenate([np.ones(len(x), int), np.zeros(len(x), int)])
    w = np.concatenate([px * eta * a1, px * (1 - eta)])
    return X, y, w

logit = lambda p: np.log(p) - np.log1p(-p)
rows = []

## E1. Mức tổng thể, mô hình bão hòa (mọi hàm f)
x nhận 5 giá trị (như C1), P(x) đều, η(x) tùy ý (không có dạng logistic). Đặc trưng one-hot → mô hình biểu diễn được mọi f.
Lý thuyết: f*_a(x) = logit η(x) + log(a1/a0) tại MỌI x.

In [2]:
xv = np.array([-2, -1, 0, 1, 2]); eta = np.array([0.10, 0.30, 0.20, 0.60, 0.85]); px = np.full(5, 0.2)
onehot = np.eye(5)
for r in RATIOS:
    X, y, w = population_rows(onehot, px, eta, r)
    X = np.vstack([onehot, onehot])
    f, m = fit_logit(X, y, w)
    err = f(onehot) - logit(eta) - np.log(r)
    rows.append(dict(thi_nghiem="E1_bao_hoa", ti_so=r, log_ti_so=np.log(r),
                     sai_so_max=np.max(np.abs(err)), hoi_tu=m.converged_, n_iter=m.n_iter_))
pd.DataFrame([r for r in rows if r["thi_nghiem"] == "E1_bao_hoa"])

,thi_nghiem,ti_so,log_ti_so,sai_so_max,hoi_tu,n_iter
0,E1_bao_hoa,0.2,-1.609438,0.000044,True,72
1,E1_bao_hoa,0.5,-0.693147,0.000019,True,58
2,E1_bao_hoa,1.0,0.000000,0.000012,True,51
3,E1_bao_hoa,2.0,0.693147,0.000006,True,50
4,E1_bao_hoa,5.0,1.609438,0.000018,True,62
5,E1_bao_hoa,10.0,2.302585,0.000046,True,76


## E2. Mức tổng thể, mô hình tuyến tính f(x) = w·x + b
x ~ N(0, 1), tích phân xấp xỉ bằng lưới 1201 điểm trên [−6, 6].
(a) Đúng dạng: η(x) = σ(1,5x − 0,5) → kỳ vọng Δb = log(a1/a0), Δw = 0.
(b) Sai dạng: η(x) = σ(2 sin(1,5x)) → không còn bảo đảm (chỉ đúng khi họ mô hình chứa f*).

In [3]:
grid = np.linspace(-6, 6, 1201); pg = np.exp(-grid**2 / 2); pg /= pg.sum()
Xg = np.concatenate([grid, grid])[:, None]
xs = np.linspace(-3, 3, 601)[:, None]
cases = {"E2a_dung_dang": core.sigmoid(1.5 * grid - 0.5), "E2b_sai_dang": core.sigmoid(2 * np.sin(1.5 * grid))}
for name, eta_g in cases.items():
    _, y, w = population_rows(grid, pg, eta_g, 1.0)
    f1, m1 = fit_logit(Xg, y, w)
    w1, b1 = m1.W_[0, 1] - m1.W_[0, 0], m1.b_[1] - m1.b_[0]
    for r in RATIOS:
        _, y, w = population_rows(grid, pg, eta_g, r)
        f, m = fit_logit(Xg, y, w)
        wr, br = m.W_[0, 1] - m.W_[0, 0], m.b_[1] - m.b_[0]
        rows.append(dict(thi_nghiem=name, ti_so=r, log_ti_so=np.log(r), w=wr, b=br,
                         delta_b=br - b1, delta_w=wr - w1,
                         sai_so_max=np.max(np.abs(f(xs) - f1(xs) - np.log(r))),
                         hoi_tu=m.converged_, n_iter=m.n_iter_))
pd.DataFrame([r for r in rows if r["thi_nghiem"].startswith("E2")]).round(6)

,thi_nghiem,ti_so,log_ti_so,w,b,delta_b,delta_w,sai_so_max,hoi_tu,n_iter
0,E2a_dung_dang,0.2,-1.609438,1.499998,-2.109440,-1.609440,-0.000002,0.000007,True,43
1,E2a_dung_dang,0.5,-0.693147,1.500000,-1.193147,-0.693147,-0.000000,0.000000,True,22
2,E2a_dung_dang,1.0,0.000000,1.500000,-0.500000,0.000000,0.000000,0.000000,True,24
3,E2a_dung_dang,2.0,0.693147,1.500000,0.193147,0.693147,0.000000,0.000000,True,22
4,E2a_dung_dang,5.0,1.609438,1.500000,1.109438,1.609438,0.000000,0.000000,True,20
5,E2a_dung_dang,10.0,2.302585,1.500000,1.802585,2.302585,0.000000,0.000000,True,26
6,E2b_sai_dang,0.2,-1.609438,0.818445,-1.586306,-1.586306,-0.120867,0.385733,True,17
7,E2b_sai_dang,0.5,-0.693147,0.910092,-0.680582,-0.680582,-0.029220,0.100224,True,24
8,E2b_sai_dang,1.0,0.000000,0.939312,0.000000,0.000000,0.000000,0.000000,True,18
9,E2b_sai_dang,2.0,0.693147,0.910092,0.680582,0.680582,-0.029220,0.100224,True,24


## E3. Mẫu hữu hạn (đúng dạng), 5 seed
Δb − log(a1/a0) và Δw theo n; đối chiếu hệ số với sklearn `class_weight={0: 1, 1: r}`.

In [4]:
e3 = []
for n in (1_000, 10_000, 100_000):
    for r in (0.2, 5.0):
        for s in SEEDS:
            rng = np.random.default_rng(s)
            x = rng.normal(size=n); y = (rng.uniform(size=n) < core.sigmoid(1.5 * x - 0.5)).astype(int)
            _, m1 = fit_logit(x[:, None], y, None)
            _, mr = fit_logit(x[:, None], y, np.where(y == 1, r, 1.0))
            db = (mr.b_[1] - mr.b_[0]) - (m1.b_[1] - m1.b_[0])
            dw = (mr.W_[0, 1] - mr.W_[0, 0]) - (m1.W_[0, 1] - m1.W_[0, 0])
            sk = LogisticRegression(C=np.inf, class_weight={0: 1, 1: r}, tol=1e-12, max_iter=10_000).fit(x[:, None], y)
            dsk = max(abs(sk.coef_[0, 0] - (mr.W_[0, 1] - mr.W_[0, 0])), abs(sk.intercept_[0] - (mr.b_[1] - mr.b_[0])))
            e3.append(dict(n=n, ti_so=r, seed=s, loi_db=db - np.log(r), dw=dw, lech_sklearn=dsk,
                           hoi_tu=m1.converged_ and mr.converged_))
e3 = pd.DataFrame(e3)
agg = e3.groupby(["n", "ti_so"]).agg(loi_db_tb=("loi_db", "mean"), loi_db_dlc=("loi_db", "std"),
                                     dw_tb=("dw", "mean"), dw_dlc=("dw", "std"),
                                     lech_sklearn_max=("lech_sklearn", "max"), hoi_tu=("hoi_tu", "all")).reset_index()
for _, a in agg.iterrows():
    rows.append(dict(thi_nghiem="E3_mau_huu_han", n=a.n, ti_so=a.ti_so, log_ti_so=np.log(a.ti_so),
                     loi_db_tb=a.loi_db_tb, loi_db_dlc=a.loi_db_dlc, dw_tb=a.dw_tb, dw_dlc=a.dw_dlc,
                     lech_sklearn_max=a.lech_sklearn_max, hoi_tu=a.hoi_tu))
agg

,n,ti_so,loi_db_tb,loi_db_dlc,dw_tb,dw_dlc,lech_sklearn_max,hoi_tu
0,1000,0.2,-0.002913,0.010620,0.027646,0.059796,1.064223e-05,True
1,1000,5.0,0.000847,0.004476,-0.026135,0.046878,1.081313e-08,True
2,10000,0.2,0.000309,0.001967,-0.004299,0.011115,4.783295e-06,True
3,10000,5.0,-0.000604,0.001171,0.007349,0.012563,1.267717e-08,True
4,100000,0.2,-0.000578,0.000497,0.001092,0.003859,2.420334e-06,True
5,100000,5.0,0.000177,0.000361,0.001921,0.005964,2.604598e-08,True


## E4. Liên hệ ngưỡng B5
c_FP = 1, c_FN = 5 → t* = 1/6. So sánh: (A) không trọng số, đoán dương khi p ≥ 1/6;
(B) trọng số a1/a0 = c_FN/c_FP = 5, đoán dương khi p ≥ 1/2; (O) quy tắc Bayes dùng η thật.
Train n = 10 000, test n = 100 000 (độc lập), 5 seed. Chi phí trung bình mỗi mẫu trên test.

In [5]:
cFP, cFN = 1.0, 5.0
e4 = []
for name, eta_fn in (("dung_dang", lambda x: core.sigmoid(1.5 * x - 0.5)),
                      ("sai_dang", lambda x: core.sigmoid(2 * np.sin(1.5 * x)))):
    for s in SEEDS:
        rng = np.random.default_rng(s)
        xtr = rng.normal(size=10_000); ytr = (rng.uniform(size=xtr.size) < eta_fn(xtr)).astype(int)
        xte = rng.normal(size=100_000); yte = (rng.uniform(size=xte.size) < eta_fn(xte)).astype(int)
        fA, _ = fit_logit(xtr[:, None], ytr, None)
        fB, _ = fit_logit(xtr[:, None], ytr, np.where(ytr == 1, cFN / cFP, 1.0))
        dA = core.sigmoid(fA(xte[:, None])) >= cFP / (cFP + cFN)
        dB = core.sigmoid(fB(xte[:, None])) >= 0.5
        dO = eta_fn(xte) >= cFP / (cFP + cFN)
        cost = lambda d: np.mean(cFP * (d & (yte == 0)) + cFN * (~d & (yte == 1)))
        e4.append(dict(truong_hop=name, seed=s, trung_A_B=np.mean(dA == dB),
                       chi_phi_A=cost(dA), chi_phi_B=cost(dB), chi_phi_bayes=cost(dO)))
e4 = pd.DataFrame(e4)
agg4 = e4.groupby("truong_hop").agg(["mean", "std"]).drop(columns="seed")
for name, a in agg4.iterrows():
    rows.append(dict(thi_nghiem="E4_nguong_B5_" + name, ti_so=cFN / cFP, log_ti_so=np.log(cFN / cFP),
                     trung_A_B_tb=a[("trung_A_B", "mean")], trung_A_B_dlc=a[("trung_A_B", "std")],
                     chi_phi_A_tb=a[("chi_phi_A", "mean")], chi_phi_A_dlc=a[("chi_phi_A", "std")],
                     chi_phi_B_tb=a[("chi_phi_B", "mean")], chi_phi_B_dlc=a[("chi_phi_B", "std")],
                     chi_phi_bayes_tb=a[("chi_phi_bayes", "mean")], chi_phi_bayes_dlc=a[("chi_phi_bayes", "std")]))
agg4

trung_A_B           chi_phi_A           chi_phi_B            \
                mean       std      mean       std      mean       std   
truong_hop                                                               
dung_dang   0.998488  0.001657  0.481446  0.002746  0.481402  0.002919   
sai_dang    0.984002  0.002534  0.585148  0.002985  0.572502  0.003607   

           chi_phi_bayes            
                    mean       std  
truong_hop                          
dung_dang       0.481146  0.002381  
sai_dang        0.461612  0.002941

In [6]:
out = pd.DataFrame(rows)
out.to_csv(os.path.join(RES, "C4_logit_shift.csv"), index=False)
print("Đã lưu results/C4_logit_shift.csv:", out.shape)

Đã lưu results/C4_logit_shift.csv: (26, 24)
